In [1]:
import sys
from types import ModuleType

# 제거된 vertexai 모듈을 메모리에 가짜로 등록
dummy_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertex.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertex

dummy_llm_vertex = ModuleType("langchain_community.llms.vertexai")
dummy_llm_vertex.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = dummy_llm_vertex


In [2]:
import pandas as pd
from ragas import EvaluationDataset # 얘를통해서 ?변환해서?평가를한다?

# 검색 실패 분석을 위해 의도적으로 품질 결함이 포함된 샘플 데이터를 정의합니다.
# 1번 케이스: 메타데이터 필터 미흡으로 과거(2023년) 문서가 검색된 사례
# 2번 케이스: 청킹 분할 실패로 운영 시간 문장이 중간에 잘려 나간 사례
# 3번 케이스: 정상적으로 검색된 성공 사례
eval_raw_data = {
    "user_input": [
        "2024년 1학기 교내 장학금 신청 마감일은 언제인가요?",
        "중앙도서관 주말 운영 시간 및 대출 규정은 어떻게 되나요?",
        "기숙사 식당 아침 식사 시간은 언제인가요?"
    ],
    "reference": [
        "2024년 1학기 교내 장학금 신청 마감일은 2024년 5월 20일 18시까지입니다.",
        "중앙도서관 주말 운영 시간은 오전 10시부터 오후 5시까지이며, 1인당 최대 5권까지 대출 가능합니다.",
        "기숙사 식당 조식 운영 시간은 오전 7시부터 오전 9시까지입니다."
    ],
    "retrieved_contexts": [
        [
            "2023년 2학기 교내 장학금 신청 마감일은 11월 30일까지입니다.",
            "장학재단 외부 장학금 접수 안내 공지사항입니다."
        ],
        [
            "중앙도서관 주말 운영 안내: 토요일과 일요일은 오전 10시부터 오후 5시까지... (문맥 끊김)",
            "도서관 매점은 평일 오전 9시부터 오후 6시까지 운영됩니다."
        ],
        [
            "기숙사 식당 운영 안내: 조식은 07:00~09:00, 중식은 11:30~13:30에 제공됩니다.",
            "기숙사 헬스장은 24시간 자율 개방됩니다."
        ]
    ]
}

df_eval = pd.DataFrame(eval_raw_data)

ragas_dataset = EvaluationDataset.from_pandas(df_eval)

ragas_dataset

c:\Users\kimmi\Documents\ax_study\12_rag\chapter06_07\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference'], len=3)

In [3]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from ragas.metrics import ContextPrecision, ContextRecall
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

load_dotenv()

model = init_chat_model(
    model="gpt-6-luna", # luna를쓰면 temperature를 머 문제가잇어서..? 충돌이.? 문제없는건 옛날모델거. 근데 갑자기 bypass저거추가하고 루나를써도되게됨.왜.
    model_provider="openai",
    reasoning_effort="none",
    temperature=0
)

evaluator_llm = LangchainLLMWrapper(model, bypass_temperature=True) # llm도잇지만 임베딩레퍼도잇다.먼가 추상화된 틀이아닌가 인터페이스기반으로,,

metrics = [
    ContextPrecision(llm=evaluator_llm), # 정밀도
    ContextRecall(llm=evaluator_llm) # 재현율 
]
# ㄴ> 정밀도, 재현율 각자의 llm모델을 따로 분리할수도있는 장점이있다. 그럼 문제강없나.?

evaluation_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    raise_exceptions=True
)

C:\Users\kimmi\AppData\Local\Temp\ipykernel_34012\545980863.py:5: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\kimmi\AppData\Local\Temp\ipykernel_34012\545980863.py:5: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\kimmi\AppData\Local\Temp\ipykernel_34012\545980863.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='..

In [4]:
evaluation_result

{'context_precision': 0.6667, 'context_recall': 0.3333}

In [5]:
df_result = evaluation_result.to_pandas()

output_csv_path = "search_eval_report.csv"
df_result.to_csv(output_csv_path, index=False, encoding="utf-8-sig")